# Les pénuries de carburant se propagent-elles comme une épidémie ?
### La crise d'octobre 2022 dans les stations-service françaises

Auteurs : Sarah LIABES, Adam Trabelsi, Eden Szezot

## Sommaire

- Introduction
- I. Données
  - A. Ruptures de carburant par station
  - B. Raffineries, dépôts et dates de grève
  - C. Sources de croisement (chiffres officiels, INSEE, contours)
  - D. Contrôles qualité et export
- II. Dynamique de la crise
- III. Choc commun ou contagion locale ?
  - A. H1 : diffusion depuis les raffineries et dépôts en grève
  - B. H2 : report des automobilistes sur les stations voisines
- IV. Comparaison avec une vraie épidémie : le COVID-19
- Conclusion et limites

## Introduction

**Contexte.** 

**Problématique.** Pendant la crise d'octobre 2022, les ruptures en stations-service s'expliquent-elles par un **choc logistique commun** (raffineries et dépôts en grève) ou par une **contagion locale**, les automobilistes se reportant sur les stations voisines ?

**Hypothèses.**
- **H1 — choc commun :** les ruptures se concentrent autour des sites en grève et les stations d'une même zone tombent en même temps.
- **H2 — contagion par report :** une station a plus de chances de tomber en rupture lorsque ses voisines le sont déjà, à approvisionnement comparable.

**Démarche.** 

### Configuration

Le code est rangé dans le package `src/penuries/`. Les cellules ci-dessous l'importent ; les données sont téléchargées depuis leurs sources officielles.

In [1]:
import sys

sys.path.append("src")

import pandas as pd

from penuries import collecte, nettoyage, indicateurs, visualisation

## I. Données

### A. Ruptures de carburant par station

_Source : flux annuel 2022 du site prix-carburants.gouv.fr (données ouvertes du ministère de l'Économie)._

Le flux annuel 2022 recense, pour chaque station-service, ses prix et ses ruptures (carburant, date de début, date de fin si elle est connue). Nous le téléchargeons directement depuis la source officielle, puis nous en extrayons deux tables :
- `df_stations` : une ligne par station (coordonnées, code postal, autoroute ou route, nombre de mises à jour de prix sur la période) ;
- `df_ruptures` : une ligne par épisode de rupture.



In [2]:
chemin_xml = collecte.telecharger_flux_annuel(2022)

df_stations = collecte.lire_stations(chemin_xml)
df_ruptures = collecte.lire_ruptures(chemin_xml)

print(f"{len(df_stations):,} stations, dont {(df_stations['nb_maj_periode'] > 0).sum():,} actives entre septembre et novembre 2022")
print(f"{len(df_ruptures):,} épisodes de rupture au total")
df_ruptures.head()

13,645 stations, dont 9,563 actives entre septembre et novembre 2022
195,032 épisodes de rupture au total


,id_station,latitude,longitude,code_postal,type_route,carburant,debut,fin
0,1000001,4620100,519800,01000,R,GPLc,2017-09-16 09:50:23,NaT
1,1000001,4620100,519800,01000,R,E10,2018-12-13 09:49:49,NaT
2,1000001,4620100,519800,01000,R,E85,2017-09-16 09:50:23,NaT
3,1000001,4620100,519800,01000,R,SP98,2022-10-21 07:51:17,2022-10-21 07:52:49
4,1000001,4620100,519800,01000,R,SP95,2022-10-11 08:31:05,2022-10-11 17:10:56


In [3]:
df_ruptures["debut"].dt.to_period("M").value_counts().sort_index().loc["2022-06":"2022-12"]

debut
2022-06     1654
2022-07     2927
2022-08     2461
2022-09    24528
2022-10    73716
2022-11    36292
2022-12    16000
Freq: M, Name: count, dtype: int64

Par exemple, les ruptures commencées en octobre 2022 sont environ 30 fois plus nombreuses qu'en août, ce qui confirme l'ampleur de la crise. Le fichier contient aussi des ruptures très anciennes sans date de fin (carburants que la station ne vend plus) : nous les traitons dans la partie D.

#### Nettoyage

Nous ne gardons que ce qui sert à répondre à la question :

- **Les stations actives de métropole :** celles qui ont déclaré au moins un prix entre septembre et novembre 2022, avec des coordonnées valides. Le département est déduit du code postal (Corse : 2A pour les codes inférieurs à 20200, 2B au-delà) ; l'outre-mer est exclu.
- **Les ruptures commencées entre le 1er septembre et le 30 novembre 2022.**
- **Les quatre carburants principaux** (gazole, SP95, SP95-E10, SP98) : c'est la définition qu'utilisait le ministère pour compter les stations « en difficulté », ce qui rend nos chiffres comparables aux siens.
- **Les ruptures d'au moins 1 heure et d'au plus 30 jours, avec une date de fin connue.** En dessous d'une heure, il s'agit plutôt d'erreurs de déclaration ; au-delà de 30 jours, ou sans fin connue au 31 décembre, la station a probablement cessé de vendre ce carburant.

In [4]:
df_stations = nettoyage.nettoyer_stations(df_stations)
df_ruptures_periode = nettoyage.nettoyer_ruptures(df_ruptures, df_stations)

print(f"{len(df_stations):,} stations actives dans {df_stations['departement'].nunique()} départements")
print(f"{len(df_ruptures_periode):,} ruptures des carburants principaux entre septembre et novembre 2022")
print(f"Part avec une date de fin connue : {df_ruptures_periode['fin_connue'].mean():.1%}")
df_ruptures_periode["duree_heures"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

9,563 stations actives dans 96 départements
113,942 ruptures des carburants principaux entre septembre et novembre 2022
Part avec une date de fin connue : 98.7%


count    112437.000000
mean         74.127808
std         130.608423
min           0.000000
5%            1.157222
25%          14.321389
50%          30.429167
75%          76.272778
95%         287.586167
max        2375.642500
Name: duree_heures, dtype: float64

In [5]:
df_ruptures_propres = nettoyage.filtrer_durees(df_ruptures_periode)

part_ecartee = 1 - len(df_ruptures_propres) / len(df_ruptures_periode)
print(f"{len(df_ruptures_propres):,} ruptures conservées ({part_ecartee:.1%} écartées par les filtres de durée)")
df_ruptures_propres["carburant"].value_counts()

106,325 ruptures conservées (6.7% écartées par les filtres de durée)


carburant
Gazole    35072
E10       31469
SP98      28955
SP95      10829
Name: count, dtype: int64

Sur les 9 563 stations actives de métropole (96 départements), nous retenons 106 325 ruptures des carburants principaux entre septembre et novembre 2022. Les filtres écartent environ 6 % des ruptures : moins de 5 % durent moins d'une heure, 1,3 % n'ont pas de date de fin connue au 31 décembre, et moins de 1 % dépassent 30 jours. Ces seuils éliminent donc les anomalies sans toucher au cœur des données.

La durée médiane d'une rupture est de 30 heures : dans la moitié des cas, une station est réapprovisionnée en un peu plus d'une journée. La distribution est cependant très étalée : un quart des ruptures dépasse 3 jours et 10 % dépassent une semaine. Ces ruptures longues pourraient concerner les stations dépendant des sites restés en grève le plus longtemps : nous le testerons dans la partie III.

Le gazole est le carburant le plus souvent en rupture, devant le SP95-E10 et le SP98. Le SP95 apparaît moins souvent, probablement parce que moins de stations le distribuent depuis la généralisation du SP95-E10 : il faudra raisonner en proportion de stations concernées, et non en nombre brut de ruptures.

### B. Raffineries, dépôts et dates de grève

_Table construite à la main (`data/manuel/`), une source citée par ligne._

### C. Sources de croisement

_Chiffres officiels quotidiens du ministère de la Transition énergétique, population INSEE par département, contours des départements._

### D. Contrôles qualité et export


#### Validation de l'indicateur

Nous définissons une station comme en rupture un jour donné si, à 13 h, elle déclare au moins une rupture en cours sur l'un des quatre carburants principaux. L'heure de 13 h est celle des relevés publiés par le ministère pendant la crise.

Avant d'analyser la crise, nous vérifions que cet indicateur est fiable de trois façons :
1. **Son niveau en temps normal** (septembre, avant les grèves) doit être bas.
2. **La date du pic** doit correspondre à celle des sources officielles.
3. **Son niveau pendant la crise** doit être comparable aux chiffres publiés : ceux du ministère de la Transition énergétique, et ceux de l'analyse indépendante de l'AFP sur les mêmes données publiques.

In [ ]:
"""niveau_normal = df_prev.set_index("jour").loc["2022-09-05":"2022-09-25", "pct_en_rupture"]
pic_prevalence = df_prev.loc[df_prev["pct_en_rupture"].idxmax()]
pic_incidence = df_prev.loc[df_prev["nouvelles_ruptures"].idxmax()]

print(f"Niveau normal (5-25 septembre) : {niveau_normal.min():.1f} % à {niveau_normal.max():.1f} %, moyenne {niveau_normal.mean():.1f} %")
print(f"Pic de prévalence : {pic_prevalence['pct_en_rupture']:.1f} % le {pic_prevalence['jour']:%d/%m}")
print(f"Pic d'incidence : {pic_incidence['nouvelles_ruptures']} nouvelles stations en rupture le {pic_incidence['jour']:%d/%m}")"""

In [ ]:
"""df_officiels = pd.read_csv("data/manuel/chiffres_officiels.csv", parse_dates=["date"])

comparaison = df_officiels.merge(
    df_prev[["jour", "pct_en_rupture"]], left_on="date", right_on="jour", how="left"
)
comparaison[["date", "source", "indicateur", "valeur", "pct_en_rupture"]].round(1)"""

En temps normal, 4 à 8 % des stations déclarent une rupture, avec de petits pics les dimanches et lundis, faute de livraison le week-end. Pendant la crise, l'indicateur atteint 49,6 % le 16 octobre, soit environ huit fois ce niveau. Le pic des nouvelles ruptures (857 stations le 10 octobre) coïncide avec le pic des chiffres officiels.

Nos pourcentages sont toutefois 6 à 20 points au-dessus de ceux du ministère, et décroissent plus lentement. Deux raisons l'expliquent :
- **La définition diffère.** Selon l'AFP, le ministère ne comptait que les stations déclarant une « rupture » et excluait celles marquées « non distribué ». Cette distinction n'existe pas dans les données publiques, qui mélangent les deux.
- **Les fins de rupture sont parfois déclarées en retard.** Une rupture reste ouverte dans le fichier tant que la station ne met pas à jour ses données. Cela gonfle la fin de la courbe et retarde le pic de prévalence.

L'analyse de l'AFP, menée sur les mêmes données publiques, trouve des niveaux proches des nôtres (environ 47 % de stations manquant d'essence ou de gazole le 17 octobre, 31 % le 20 octobre). Nos chiffres sont un peu au-dessus, car une station compte chez nous dès qu'il lui manque un seul des quatre carburants.

Conséquence pour la suite : notre indicateur surestime probablement le niveau absolu de la pénurie, mais il en restitue bien la chronologie. Nous l'utilisons donc surtout pour comparer des dates, des départements et des stations entre eux, ce qui est exactement ce que demande notre question.